# Step 19 — Inference, 10-model blend (160px + 224px)

**Settings: `GPU T4 x2`, `Internet OFF`.**

**Inputs:**
1. competition data
2. `rsna-knee-checkpoints` (the 224px models, rebuilt from parts)
3. `rsna-knee-05-train-fold0` (160px fold 0)
4. `rsna-knee-08-train-folds` (160px folds 1-4)

## The blend

Validated offline on 4,216 out-of-fold studies:

| | OOF macro |
|---|---|
| 224px alone | 0.8020 |
| **blend, w224 = 0.60** | **0.8090** (+0.0070) |
| per-label weights | 0.8094 (+0.0003 more — not worth 12 fitted parameters) |

So: `0.40 x mean(160px models) + 0.60 x mean(224px models)`, plain mean averaging.

## Each model gets the preprocessing it was trained on

This matters and it's easy to get wrong. The two model families were trained on **different** pipelines:

| | 160px models | 224px models |
|---|---|---|
| resolution | 160 | 224 |
| slices | 24 | 32 |
| laterality | tag only — **unmirrored when the tag was missing** | tag, then geometric fallback (100% resolved) |

The 160px models learned on data where ~24% of studies had medial and lateral reversed. Feeding them corrected images now would be a train/test mismatch. So the notebook runs **two decoders**, and each family sees exactly the distribution it was trained on. That's also precisely what the OOF measurement above was based on.

## Runtime

~99 minutes: 21 min decoding both resolutions, 20 min for the five 160px models, 58 min for the five 224px. Against a 9-hour limit.

### 1 · Setup

In [ ]:
import os, gc, glob, time, zipfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom, torch, torch.nn as nn
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
W224 = 0.60                       # from the offline blend search
LAT_THR, LAT_SIGN = -37.5, 1

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
test = pd.read_csv(f"{ROOT}/test.csv")
tser = pd.read_csv(f"{ROOT}/test_series.csv")
studies = test["StudyInstanceUID"].tolist()
print(f"test studies: {len(studies):,}  |  w224 = {W224}")

### 2 · Load both model families

The 224px checkpoints are rebuilt from the unpacked dataset; the 160px ones load directly from notebook outputs.

In [ ]:
# --- 224px: rebuild from the unpacked dataset ---
roots = [p for p in glob.glob("/kaggle/input/*") if not p.endswith("competitions")]
pkls = []
for r in roots:
    pkls += glob.glob(f"{r}/**/data.pkl", recursive=True)

def rebuild(src, out):
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, d, files in os.walk(src):
            for n in files:
                full = os.path.join(r, n)
                rel  = os.path.relpath(full, src).replace("\\", "/")
                z.write(full, "archive/" + rel)
    return out

CK224 = []
for pkl in sorted(pkls):
    src = os.path.dirname(pkl)
    tag = [p for p in src.split("/") if "fold" in p.lower()]
    name = (tag[-1] if tag else f"ck{len(CK224)}") + ".pt"
    CK224.append(rebuild(src, f"/kaggle/working/{name}"))
print(f"224px checkpoints rebuilt: {len(CK224)}")

# --- 160px: straight from notebook outputs ---
CK160 = sorted(glob.glob("/kaggle/input/notebooks/*/*/model_fold*.pt"))
print(f"160px checkpoints found  : {len(CK160)}")
assert CK224 and CK160, "missing one of the model families"

In [ ]:
import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        return (x * a.softmax(-1).unsqueeze(-1)).sum(1)

class Net(nn.Module):
    """gpu25d=True  -> input (B, S, T, H, W),    2.5D stack built here  (224px models)
       gpu25d=False -> input (B, S, T, 3, H, W), 2.5D already stacked   (160px models)"""
    def __init__(self, nlab=12, hid=256, gpu25d=True):
        super().__init__()
        self.gpu25d = gpu25d
        self.enc = timm.create_model("efficientnet_b0", pretrained=False,
                                     num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))

    def forward(self, v, mask):
        B, S, T = v.shape[0], v.shape[1], v.shape[2]
        if self.gpu25d:
            H, W = v.shape[3], v.shape[4]
            lo = torch.cat([v[:, :, :1], v[:, :, :-1]], 2)
            hi = torch.cat([v[:, :, 1:], v[:, :, -1:]], 2)
            x  = torch.stack([lo, v, hi], 3).reshape(B * S * T, 3, H, W)
        else:
            H, W = v.shape[4], v.shape[5]
            x = v.reshape(B * S * T, 3, H, W)

        f = self.enc(x).reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        return self.head(self.ser_pool(f, mask))


def load_models(paths, gpu25d):
    ms = []
    for p in paths:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        m = Net(gpu25d=gpu25d).cuda(); m.load_state_dict(ck["model"]); m.eval()
        ms.append(m)
        print(f"  fold {ck['fold']}: val {ck['val_auc']:.4f} gold {ck['gold_auc']:.4f}")
    return ms

print("224px models:"); M224 = load_models(CK224, True)
print("160px models:"); M160 = load_models(CK160, False)


### 3 · Two decoders

`decode(dirpath, px, nsl, fix_lat)` — one function, two configurations.

`fix_lat=False` reproduces the original 160px pipeline: mirror only when a `Laterality` tag exists. `fix_lat=True` adds the geometric fallback so every study is resolved, matching the 224px cache.

In [ ]:
def pick(g):
    out = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

chosen = {suid: pick(g) for suid, g in tser.groupby("StudyInstanceUID")}


def decode(dirpath, px, nsl, fix_lat):
    files = os.listdir(dirpath)
    if not files: return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas: return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]; r, c = iop[:3], iop[3:]; n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos); metas = [metas[i] for i in order]
    take = np.linspace(0, len(metas)-1, nsl).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((px, px), np.float32)
        imgs.append(cv2.resize(a, (px, px), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)

    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    axes = [n, c, r]
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0: vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    tag = next((m[4] for m in metas if m[4]), "")
    if tag:
        side = tag.strip().upper()[0]
    elif fix_lat:
        xs = [float(m[2][0]) for m in metas if m[2] is not None]
        side = ("L" if (np.mean(xs) - LAT_THR) * LAT_SIGN > 0 else "R") if xs else "R"
    else:
        side = ""                      # original 160px behaviour: no mirroring
    if side == "L":
        vol = np.flip(vol, axis=int(np.argmax([abs(v[0]) for v in axes])))

    return (vol * 255).astype(np.uint8)


def load_study(i):
    suid = studies[i]
    a160 = np.zeros((3, 24, 160, 160), np.uint8)
    a224 = np.zeros((3, 32, 224, 224), np.uint8)
    pres = np.zeros(3, np.float32)
    for k, cs in enumerate(chosen.get(suid, [None]*3)):
        if cs is None: continue
        p = f"{ROOT}/test_series/{suid}/{cs}"
        if not os.path.isdir(p): continue
        try:
            v1 = decode(p, 160, 24, False)
            v2 = decode(p, 224, 32, True)
            if v1 is not None: a160[k] = v1
            if v2 is not None: a224[k] = v2; pres[k] = 1.0
        except Exception:
            pass
    return i, a160, a224, pres

print("decoders ready")

### 4 · Predict and blend

In [ ]:
CHUNK, BS224, BS160 = 64, 3, 6
preds = np.full((len(studies), 12), 0.5, np.float32)

t0 = time.time()
for c0 in range(0, len(studies), CHUNK):
    idx = list(range(c0, min(c0 + CHUNK, len(studies))))
    with Pool(4) as pool:
        res = pool.map(load_study, idx)

    ids  = [r[0] for r in res]
    v160 = np.stack([r[1] for r in res]).astype(np.float32) / 255.0
    v224 = np.stack([r[2] for r in res]).astype(np.float32) / 255.0
    mask = np.stack([r[3] for r in res])
    B = len(ids)

    # 160px models need the 2.5D stack materialised: (B, S, T, 3, H, W)
    x160 = np.empty((B, 3, 24, 3, 160, 160), np.float32)
    for b in range(B):
        for s in range(3):
            for j in range(24):
                lo, hi = max(j-1, 0), min(j+1, 23)
                x160[b, s, j] = np.stack([v160[b, s, lo], v160[b, s, j], v160[b, s, hi]])

    out = np.zeros((B, 12), np.float32)
    with torch.no_grad():
        for b0 in range(0, B, BS160):
            xb = torch.from_numpy(x160[b0:b0+BS160]).cuda()
            mb = torch.from_numpy(mask[b0:b0+BS160]).cuda()
            acc = None
            for m in M160:
                with torch.amp.autocast("cuda"):
                    o = m(xb, mb).float().sigmoid()
                acc = o if acc is None else acc + o
            out[b0:b0+BS160] += (1 - W224) * (acc / len(M160)).cpu().numpy()

        for b0 in range(0, B, BS224):
            xb = torch.from_numpy(v224[b0:b0+BS224]).cuda()
            mb = torch.from_numpy(mask[b0:b0+BS224]).cuda()
            acc = None
            for m in M224:
                with torch.amp.autocast("cuda"):
                    o = m(xb, mb).float().sigmoid()
                acc = o if acc is None else acc + o
            out[b0:b0+BS224] += W224 * (acc / len(M224)).cpu().numpy()

    preds[ids] = out
    del v160, v224, x160; gc.collect()
    done = c0 + len(idx); el = time.time() - t0
    print(f"  {done}/{len(studies)}  {el/done:.2f}s/study  eta {(len(studies)-done)*el/done/60:.0f} min", flush=True)

print(f"\ndone in {(time.time()-t0)/60:.1f} min")

### 5 · Write submission

In [ ]:
sub = pd.DataFrame(preds, columns=LABELS)
sub.insert(0, "StudyInstanceUID", studies)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
print(f"rows left at default 0.5: {(preds == 0.5).all(1).sum()}")

for p in CK224:                      # keep only submission.csv in the output
    try: os.remove(p)
    except Exception: pass
print("cleaned up rebuilt checkpoints")